## What is an LLM Gateway?:
 - Think of an **LLM Gateway** as a **smart middleware layer** that sits between your application and multiple LLM providers (OpenAI, Anthropic, Google, Groq, Cohere, local models, etc.).

```
                    ┌─────────────────────────────┐
                    │       Your Application      │
                    │  (Chatbot, RAG, Agent, etc) │
                    └──────────────┬──────────────┘
                                   │
                                   ▼
                    ┌─────────────────────────────┐
                    │       LLM GATEWAY           │
                    │  • Routing                  │
                    │  • Fallbacks                │
                    │  • Caching                  │
                    │  • Rate Limiting            │
                    │  • Cost Tracking            │
                    │  • Observability            │
                    └──────┬─────┬─────┬─────┬────┘
                           │     │     │     │
                           ▼     ▼     ▼     ▼
                        OpenAI Claude Gemini Groq
```

### Without a Gateway:

- Different SDKs and APIs for every provider
- No fallback if one provider goes down
- No central place to track costs
- Hard to switch models without rewriting code
- No caching → paying twice for the same query

### With a Gateway:

- **One unified API** for 100+ providers
- **Automatic fallbacks** if a provider fails
- **Centralized logging, cost tracking, rate limiting**
- **Swap models with a config change**, no code rewrite
- **Cache repeated queries** → save money

### Installation & Setup:
We'll use:

- **LiteLLM** → the most popular open-source LLM gateway (supports 100+ providers)
- **LangChain** → for building agentic workflows on top of the gateway
- **python-dotenv** → for managing API keys

In [1]:
import warnings
import logging

warnings.filterwarnings("ignore")
logging.getLogger("Litellm").setLevel(logging.ERROR)


#now LiteLLM normally imports the following modules:
from litellm import completion




In [2]:
import litellm
litellm.suppress_debug_info = True

In [3]:
import warnings
import logging

#keep the recording clean- suppress noisy Aws related warnings:
warnings.filterwarnings("ignore")
logging.getLogger("Litellm").setLevel(logging.ERROR)

In [ ]:
# Load API keys from a .env file
# Create a .env file in the same folder with:
# OPENAI_API_KEY=sk-...
# ANTHROPIC_API_KEY=sk-ant-...
# GROQ_API_KEY=gsk_

import os
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["ANTHROPIC_API_KEY"] = os.getenv("ANTHROPIC_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")  

### LiteLLM Example — Unified API:
- The biggest pain point: every provider has a different SDK.

LiteLLM gives you one function — completion() — that works with all of them. Look at how clean this is:

In [ ]:
from litellm import completion

# same code, different providers - just change the model string:

#call OpenAI:
response_openai = completion(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Explain RAG in one sentence."}]
)
print("OpenAI:", response_openai.choices[0].message.content)

# Call Groq (super fast inference):
response_groq = completion(
    model="groq/llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Explain attention mechanism."}]
)
print("Groq:", response_groq.choices[0].message.content)


 Notice: Same code, three different providers. This alone is huge — you can switch providers with a string change.

But a real LLM Gateway does much more. Let's build those features one by one. 

In [ ]:
from litellm import completion

prompt = "Explain RAG in one sentence."

# Just a list of model strings — that's the only configuration
providers = [
    ("OpenAI",     "gpt-4o-mini"),
    (" Groq",       "groq/llama-3.3-70b-versatile"),
    (" Anthropic",  "claude-3-5-haiku-20241022"),
    (" Gemini",     "gemini/gemini-1.5-flash"),
]

# ONE loop. ONE function call. Multiple providers.
for label, model in providers:
    try:
        r = completion(model=model, messages=[{"role": "user", "content": prompt}])
        print(f"{label:<15}: {r.choices[0].message.content[:80]}")
    except Exception as e:
        print(f"{label:<15}: {type(e).__name__}")

### Automatic Fallbacks — When OpenAI Goes Down
Real story: OpenAI had a 4-hour outage in November 2023. Apps that hard-coded gpt-4 went completely dark.

With a gateway, if one provider fails, we automatically fall back to another. Production apps must have this.

In [ ]:
from litellm import completion

#define fallback chain: try GPT first, then Groq, then claude.
response = completion(
    model= "gemini/gemini-1.5-flash",
    messages=[{"role": "user", "content": "Explain RAG in one sentence"}]
    fallbacks = [
        "gpt-4o-mini",
        "groq/llama-3.3-70b-versatile",
    ]

)

print("Response: ", response.choices[0].message.content[:200], "...")
print("\n which model actually produced the response: ", response.model)



Which model actually answered? gpt-4o-mini-2024-07-18
If gpt-4o-mini is rate-limited or down, LiteLLM transparently retries with Claude, then Groq. Your app never sees the failure.

This is the #1 reason teams adopt an LLM Gateway.

In [ ]:
from litellm import completion

# Force the primary to fail by using a fake model name
# Then watch the fallback chain rescue the call
response = completion(
    model="openai/fake-nonexistent-model-9999",     # 👈 will fail intentionally
    messages=[{"role": "user", "content": "What is an LLM Gateway?"}],
    fallbacks=[
        "gpt-4o-mini",                              # 1st backup: real OpenAI model
        "groq/llama-3.3-70b-versatile"              # 2nd backup: Groq
    ]
)

print("App still got a response, even though the primary failed!")
print(f"\nModel that actually answered: {response.model}")

### Cost Tracking — Know Where Your Money Goes
LiteLLM automatically calculates the cost of every call using its built-in pricing database. No more surprise bills.

In [ ]:
from litellm import completion, completion_cost

response = completion(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Write a haiku about AI."}]
)

# Get the exact USD cost of this single call
cost = completion_cost(completion_response=response)

print("Response:    ", response.choices[0].message.content)
print("\nInput tokens: ", response.usage.prompt_tokens)
print("Output tokens:", response.usage.completion_tokens)
print(f"Cost:         ${cost:.8f}")

### Caching — Don't Pay Twice for the Same Question:
If 100 users ask "What is RAG?", you don't need to call the LLM 100 times.

Enable in-memory caching with one line:

In [4]:
import litellm

#reset any callbacks/strategies left over from previous cells:
litellm.callbacks = []
litellm.success_callbacks = []
litellm.failure_callbacks = []
litellm._async_success_callbacks = []
litellm._async_failure_callbacks = []

#also clear any router strategy state:
litellm.cache = None

print("LiteLLM state reset - ready for clean chaching demo")

LiteLLM state reset - ready for clean chaching demo


In [ ]:
import litellm
import time
from litellm import completion
from litellm.caching import  Cache

#enable in-memory caching(you can also use redis in production):
litellm.cache = Cache(type ="local")

prompt = "Write about attention mechanism in one paragraph."

#first call actually hits OpenAI:
start = time.time()
response1 = completion(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}]
    caching = True
)

t1 = time.time() - start
print(f"First API call: {t1:.2f}s - {r1.choices[0].message.content}...")

#second call --served from cache, so should be much faster:
start = time.time()
response2 = completion(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}]
    caching = True
)

t2 = time.time() - start
print(f"Second API call: {t2:.4f}s - {r2.choices[0].message.content}...")
print(f"Cache speedup: {t1/t2:.2f}x faster than first call and zero cost on second call!")

### Smart Routing — The Right Model for the Right Job:
Why use one model for everything?

 - Coding tasks → Claude Sonnet
 - Cheap summaries → GPT-4o-mini
 - Super fast replies → Groq Llama
 - Complex reasoning → Claude Opus

Use LiteLLM's Router to define routing rules:

In [ ]:
import os
from litellm import Router

model_list = [
    {
        "model_name": "fast-cheap",
        "litellm_params": {
            "model": "groq/llama-3.3-70b-versatile",
            "api_key": os.getenv("GROQ_API_KEY")
        }
    },
    {
        "model_name": "smart-coding",   # alias kept
        "litellm_params": {
            "model": "gpt-4o",  #mapped to OpenAI instead
            "api_key": os.getenv("OPENAI_API_KEY")
        }
    },
    {
        "model_name": "balanced",
        "litellm_params": {
            "model": "gpt-4o-mini",
            "api_key": os.getenv("OPENAI_API_KEY")
        }
    }
]

router = Router(model_list=model_list)

fast_response = router.completion(
    model="fast-cheap",
    messages=[{"role": "user", "content": "Summarize: AI is changing software."}]
)

code_response = router.completion(
    model="smart-coding",
    messages=[{"role": "user", "content": "Write a Python function to reverse a string."}]
)

print("Fast/cheap (Groq): ", fast_response.choices[0].message.content[:150])
print("\n Smart/coding (GPT-4o):\n", code_response.choices[0].message.content[:300])

### Load Balancing Across Multiple API Keys:
 - Hit rate limits on one OpenAI key? Add more keys to the same alias — the router load-balances automatically.

In [ ]:
from litellm import Router
import os

# Two deployments under the same alias
# A pool of "smart" models — all equally capable, just different providers
model_list = [
    {
        "model_name": "gpt-pool",
        "litellm_params": {
            "model": "gpt-4o",
            "api_key": os.getenv("OPENAI_API_KEY"),
        },
        "model_info": {"id": "openai-gpt4o"}
    },
    
    {
        "model_name": "gpt-pool",
        "litellm_params": {
            "model": "groq/llama-3.3-70b-versatile",
            "api_key": os.getenv("GROQ_API_KEY"),
        },
        "model_info": {"id": "groq-llama-70b"}
    },
]

router = Router(
    model_list=model_list,
    routing_strategy="simple-shuffle"
)

print(f"{'Request':<10}{'Deployment Picked':<22}{'Latency':<12}{'Response':<40}")
print("-" * 84)

for i in range(6):
    r = router.completion(
        model="gpt-pool",
        messages=[{"role": "user", "content": f"Say hello, request {i+1}"}]
    )
    # Pull out which deployment served this request
    deployment_id = r._hidden_params.get("model_id", "unknown")
    latency = r._response_ms
    answer = r.choices[0].message.content[:35]
    print(f"#{i+1:<9}{deployment_id:<22}{latency:>6.0f} ms   {answer}")

### Strategy 1: least-busy —
 - The "Express Checkout" PatternThe idea: Like picking the shortest line at a supermarket. The router tracks how many requests are currently in flight to each deployment and sends the new request to whichever one is least busy.

In [ ]:
import os
from litellm import Router
from collections import Counter

model_list = [
    {"model_name": "chat",
     "litellm_params": {"model": "gpt-4o-mini",
                        "api_key": os.getenv("OPENAI_API_KEY")},
     "model_info": {"id": "OpenAI"}},
    {"model_name": "chat",
     "litellm_params": {"model": "groq/llama-3.3-70b-versatile",
                        "api_key": os.getenv("GROQ_API_KEY")},
     "model_info": {"id": " Groq"}},
]

router = Router(
    model_list=model_list,
    routing_strategy="least-busy"   #the magic
)

hits = Counter()
for i in range(8):
    r = router.completion(
        model="chat",
        messages=[{"role": "user", "content": f"Say 'OK' #{i}"}],
        max_tokens=5
    )
    hits[r._hidden_params.get("model_id", "?")] += 1
    print(f"Request {i+1} → {r._hidden_params.get('model_id', '?')}")

print("\n Distribution:")
for k, v in hits.most_common():
    print(f"   {k}: {'█' * v} ({v})")

### Strategy 2: latency-based-routing —
 - The "Always Pick the Fastest" Pattern The idea: The router measures the response time of each deployment over recent calls and sends new requests to whichever has been fastest. Speed wins

In [ ]:
import os
from litellm import Router
import time

model_list = [
    {"model_name": "chat",
     "litellm_params": {"model": "gpt-4o-mini",
                        "api_key": os.getenv("OPENAI_API_KEY")},
     "model_info": {"id": "OpenAI GPT-4o-mini"}},
    {"model_name": "chat",
     "litellm_params": {"model": "groq/llama-3.3-70b-versatile",
                        "api_key": os.getenv("GROQ_API_KEY")},
     "model_info": {"id": "Groq Llama-3.3"}},
    
]

router = Router(
    model_list=model_list,
    routing_strategy="latency-based-routing"   #picks the fastest
)

# Send 10 requests and watch which deployments get picked over time
print(f"{'Req':<6}{'Deployment':<32}{'Latency':<10}")
print("-" * 50)

for i in range(10):
    start = time.time()
    r = router.completion(
        model="chat",
        messages=[{"role": "user", "content": "Reply with exactly: OK"}],
        max_tokens=5
    )
    latency_ms = (time.time() - start) * 1000
    deployment = r._hidden_params.get("model_id", "?")
    print(f"#{i+1:<5}{deployment:<32}{latency_ms:>6.0f} ms")

### Observability — Log Every Single Call
 - In production, you must log every LLM call: prompt, response, latency, cost, user_id, etc.

 LiteLLM supports custom callbacks — here's a simple logger:

In [ ]:
import litellm
from litellm import completion

# A simple in-memory log store
call_logs = []

def log_success(kwargs, completion_response, start_time, end_time):
    """Called automatically after every successful LLM call."""
    call_logs.append({
        "model": kwargs.get("model"),
        "prompt": kwargs["messages"][-1]["content"][:60],
        "input_tokens": completion_response.usage.prompt_tokens,
        "output_tokens": completion_response.usage.completion_tokens,
        "latency_sec": round((end_time - start_time).total_seconds(), 2),
        "cost_usd": kwargs.get("response_cost", 0),
        "user": kwargs.get("user", "anonymous")
    })

def log_failure(kwargs, completion_response, start_time, end_time):
    print("Call failed:", kwargs.get("exception"))

# Register the callbacks
litellm.success_callback = [log_success]
litellm.failure_callback = [log_failure]

# Make a few tagged calls
for q, user in [
    ("What is RAG?", "krish"),
    ("Explain transformers.", "student_42"),
    ("What is fine-tuning?", "krish"),
]:
    completion(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": q}],
        user=user  # tag the call for attribution
    )

# Review the audit log
import json
print(json.dumps(call_logs, indent=2, default=str))

## Integrating the Gateway with LangChain:
Here's where it really clicks for production GenAI apps:

LangChain for the orchestration (agents, chains, RAG) + LiteLLM as the unified LLM backend.

LangChain has a built-in ChatLiteLLM wrapper — drop it in like any other chat model.

In [ ]:
from langchain_litellm import ChatLiteLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Build a chat model that talks through LiteLLM
llm = ChatLiteLLM(model="gpt-4o-mini", temperature=0.3)

# A standard LangChain prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI tutor named KrishGPT. Be concise."),
    ("user", "{question}")
])

# Compose with LCEL — same syntax as native LangChain
chain = prompt | llm | StrOutputParser()

answer = chain.invoke({"question": "What is an LLM Gateway in 3 bullets?"})
print(answer)

### A Real Example — Multi-Provider LangChain Chain with Fallbacks
Let's combine everything: a LangChain chain that uses Claude as primary, with GPT and Groq as fallbacks — and logs every call.

In [ ]:
from langchain_litellm import ChatLiteLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Primary model
primary = ChatLiteLLM(model="gpt-x")

# Fallbacks (any LangChain-compatible model)
fallback_1 = ChatLiteLLM(model="gpt-4o-mini", temperature=0.2)
fallback_2 = ChatLiteLLM(model="groq/llama-3.3-70b-versatile", temperature=0.2)

# LangChain's .with_fallbacks() chains them together
robust_llm = primary.with_fallbacks([fallback_1, fallback_2])

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert AI engineer. Always reply in JSON: {{\"answer\": ...}}"),
    ("user", "{question}")
])

chain = prompt | robust_llm | StrOutputParser()

result = chain

### A Mini End-to-End Demo — Smart Router for a Chatbot:
Let's build a tiny task-aware chatbot that:

 - Decides what kind of question the user is asking (code, summary, general)
 - Routes to the right model accordingly
 - Falls back if the chosen model fails
 - Logs cost and latency

In [ ]:
import time
from litellm import completion, completion_cost

def classify_task(user_query: str) -> str:
    """Cheap classifier — uses the fastest model to decide routing."""
    cls = completion(
        model="groq/llama-3.3-70b-versatile",
        messages=[{
            "role": "user",
            "content": (
                f"Classify the following query into EXACTLY one word: "
                f"'code', 'summary', or 'general'. Query: {user_query}\n\nAnswer:"
            )
        }],
        max_tokens=5
    )
    return cls.choices[0].message.content.strip().lower()


def call_with_fallbacks(model_chain, messages):
    """Try each model in order; return the first one that succeeds."""
    last_error = None
    for model in model_chain:
        try:
            return completion(model=model, messages=messages)
        except Exception as e:
            print(f"   ⚠️  {model} failed ({type(e).__name__}), trying next...")
            last_error = e
            continue
    raise last_error


def smart_chat(user_query: str):
    """Routes to the right model based on task type, with fallbacks."""
    task = classify_task(user_query)

    # Each entry is a FULL chain: [primary, fallback1, fallback2, ...]
    # Every model name includes its provider prefix (groq/, anthropic/, etc.)
    routing = {
        "code":    ["gpt-4o",                     "gpt-4o-mini",   "groq/llama-3.3-70b-versatile"],
        "summary": ["gpt-4o-mini",                "groq/llama-3.3-70b-versatile"],
        "general": ["groq/llama-3.3-70b-versatile", "gpt-4o-mini"],
    }
    model_chain = routing.get(task, routing["general"])

    start = time.time()
    response = call_with_fallbacks(
        model_chain=model_chain,
        messages=[{"role": "user", "content": user_query}]
    )
    latency = time.time() - start

    try:
        cost = completion_cost(completion_response=response)
        cost_str = f"${cost:.6f}"
    except Exception:
        cost_str = "n/a"

    return {
        "detected_task": task,
        "model_used":    response.model,
        "answer":        response.choices[0].message.content,
        "latency_sec":   round(latency, 2),
        "cost_usd":      cost_str
    }


# Try it on three very different queries
queries = [
    "Write a Python function to compute Fibonacci numbers.",
    "Summarize the importance of attention mechanism in 2 sentences.",
    "Tell me a fun fact about elephants."
]

for q in queries:
    print("=" * 70)
    print("❓ Q:", q)
    result = smart_chat(q)
    print(f" Task:    {result['detected_task']}")
    print(f" Model:    {result['model_used']}")
    print(f" Latency: {result['latency_sec']}s")
    print(f" Cost:    {result['cost_usd']}")
    print(f" Answer:  {result['answer'][:200]}...")